# IT 520 — How much can the dispatch service remember?
**Friday, October 23, 2026 · Individual, asynchronous · About 50 minutes, including setup and submitting**

Complete this in **Google Colab** and submit the **shared URL of your own completed Colab notebook** to **Canvas by the deadline shown there**.

1. [Open the activity in Google Colab](https://colab.research.google.com/github/blkfin/it520/blob/main/activities/dispatch-memory-20261023/activity-dispatch-memory.ipynb), then choose **File → Save a copy in Drive**. Put your name in the title and work in your own saved copy.
2. Run cells **in order** with the ▶ button beside each code cell, and wait for each to finish. The supplied code builds and measures everything. You only change **settings**: a policy, a yes/no, or a number. You do not write a data structure.
3. Double-click each **Your answer** text cell, replace its `[replace]` placeholders, then press Shift+Enter. **Write each prediction before you run the cell it predicts.** A wrong prediction is fine; explaining what overturned it is the point.
4. At the end, save your copy with your answers and outputs, click **Share**, set **General access → Anyone with the link → Viewer**, and copy the link. If your account restricts link sharing, share directly with your instructor as a Viewer instead. Submit **your saved copy's URL** in Canvas; do not submit the starter link above.

If Colab disconnects, reconnect and rerun the code cells in order, including your trial cells; your written answers stay in your saved copy. Do not use "Run all".

## The job

A small delivery company runs a **dispatch service** that keeps live information about parcels: where each one is and what happened to it. Every scan sends the service a **status update**: *label created*, *picked up*, *at depot*, *out for delivery*, *delivered*. The service has a fixed **memory limit of 10 MB** for this information.

The company has to decide **what to keep**: every update or only the newest one, with or without a proof-of-delivery payload (a stand-in for the doorstep photo), and for how many parcels at once. Today you run the experiments that decide it.

Everything here is **synthetic**: authored parcel IDs (`P000000`, `P000001`, …), times and byte payloads. No real customers, addresses or photos, and no claims about any real dispatch platform.

## Recap from Wednesday (read, nothing to answer)

Wednesday's budget, in one worked line: a feed of **80,000 updates** about **20,000 parcels** has 20,000 distinct parcels, the set W. If each kept entry were 64 B of payload, **B = |W| × 64 B = 1,280,000 B = 1.28 MB**, and Fits(limit) ⇔ B ≤ 10,000,000 B is **true**.

That is a count on paper. Today you **measure** what Python actually allocates, and you change the system to see what moves the number. The measured bytes say whether this program's state fits the 10 MB limit. They say nothing about cache, speed or a real company's servers.

**Click ▶ on the toolkit cell below.** It loads the synthetic feed generator and the `trial` function you will use all session. Each trial rebuilds the state from nothing, measures it with Python's `tracemalloc`, frees it, and prints one row. It also tells you which earlier trials differ from it in **exactly one** setting, because that is the only kind of comparison that lets you blame the change on that setting.

In [ ]:
#@title Dispatch toolkit — click ▶ (you do not need to read this code)
import gc
import platform
import tracemalloc

LIMIT_BYTES = 10_000_000        # the dispatch service's memory limit: 10 MB
PROOF_BYTES = 1_000             # one synthetic proof-of-delivery payload (stands in for a photo)
BASELINE = dict(parcels=20_000, updates=4, policy="every", proof=False)
STEPS = ("label created", "picked up", "at depot", "out for delivery")
TRIALS = {}                     # every trial you run, by name, so you can compare them


def feed(parcels, updates, proof):
    """Synthetic status updates, oldest first. Each parcel's last update is 'delivered'."""
    for k in range(updates):
        last = k == updates - 1
        for p in range(parcels):
            status = "delivered" if last else STEPS[min(k, 3)]
            minute = 480 + 15 * k + p % 15             # minutes after midnight
            yield (f"P{p:06d}", status, minute, bytes(PROOF_BYTES) if (proof and last) else None)


def keep(updates_iter, policy):
    """Build the dispatch state. 'every' keeps a list of all updates; 'latest' keeps one entry per parcel."""
    if policy == "every":
        state = []
        for u in updates_iter:
            state.append(u)
    else:
        state = {}
        for u in updates_iter:
            state[u[0]] = u                               # overwrites the parcel's older status
    return state


def _check(parcels, updates, policy, proof):
    if not isinstance(parcels, int) or isinstance(parcels, bool) or not 1_000 <= parcels <= 40_000:
        raise ValueError("Set parcels to a whole number from 1000 to 40000 (no quotes, no commas).")
    if not isinstance(updates, int) or isinstance(updates, bool) or not 2 <= updates <= 12:
        raise ValueError("Set updates to a whole number from 2 to 12.")
    if policy not in ("every", "latest"):
        raise ValueError('Set policy to "every" or "latest" (with the quotes).')
    if not isinstance(proof, bool):
        raise ValueError("Set proof to True or False (capital letter, no quotes).")


def measure(parcels, updates, policy, proof):
    """Rebuild the state from scratch, measure what Python allocated to keep it, then free it."""
    _check(parcels, updates, policy, proof)
    gc.collect()
    tracemalloc.start()
    before = tracemalloc.get_traced_memory()[0]
    state = keep(feed(parcels, updates, proof), policy)
    measured = tracemalloc.get_traced_memory()[0] - before
    tracemalloc.stop()
    entries = len(state)
    proofs = sum(1 for u in (state if policy == "every" else state.values()) if u[3] is not None)
    del state
    gc.collect()                                          # nothing carries over into the next trial
    return {"entries": entries, "proof_payload": proofs * PROOF_BYTES, "measured": measured}


def trial(name, parcels, updates, policy, proof, quiet=False):
    """Run one trial, print its row, and say which earlier trials differ from it in exactly one setting."""
    settings = dict(parcels=parcels, updates=updates, policy=policy, proof=proof)
    result = measure(**settings)
    TRIALS[name] = dict(settings, **result)
    if quiet:
        return TRIALS[name]
    fits = result["measured"] <= LIMIT_BYTES
    print(f"Trial {name}: parcels={parcels:,} updates={updates} policy={policy!r} proof={proof}")
    print(f"  entries kept:            {result['entries']:>12,}")
    print(f"  proof payload kept:      {result['proof_payload']:>12,} B   (counted on paper)")
    print(f"  measured, Python allocated: {result['measured']:>9,} B = {result['measured'] / 1e6:.2f} MB")
    print(f"  measured per parcel:     {result['measured'] / parcels:>12,.0f} B")
    print(f"  Fits(limit): {result['measured']:,} B <= {LIMIT_BYTES:,} B  ->  {fits}")
    single = []
    for other, row in TRIALS.items():
        if other == name:
            continue
        changed = [k for k in settings if row[k] != settings[k]]
        if len(changed) == 1:
            k = changed[0]
            single.append(f"vs trial {other}: only {k} changed ({row[k]!r} -> {settings[k]!r})")
    if single:
        print("  One-factor comparisons: " + "; ".join(single))
    elif len(TRIALS) > 1:
        print("  No earlier trial differs from this one in only one setting. Change one thing at a time.")
    return TRIALS[name]


def show_trials():
    """Print every trial so far as one table."""
    print(f"{'trial':<6}{'parcels':>8}{'upd':>5}  {'policy':<7}{'proof':<6}{'entries':>9}{'measured MB':>13}{'B/parcel':>10}  fits")
    for name, r in TRIALS.items():
        print(f"{name:<6}{r['parcels']:>8,}{r['updates']:>5}  {r['policy']:<7}{str(r['proof']):<6}"
              f"{r['entries']:>9,}{r['measured'] / 1e6:>13.2f}{r['measured'] / r['parcels']:>10,.0f}  "
              f"{r['measured'] <= LIMIT_BYTES}")

print(f"Toolkit ready. Python {platform.python_version()}. Limit: {LIMIT_BYTES:,} B. Proof payload: {PROOF_BYTES:,} B.")

## 1. Two ways to remember parcels

The service can store updates in one of two ways. These are the **policy** setting.

- **`"every"`, keep every update.** The service appends each update to a list, so it holds the parcel's whole history. The list grows with every scan.
- **`"latest"`, keep only the latest status.** The service keeps one entry per parcel, keyed by parcel ID: a mapping from parcel to its newest update. Each new update **replaces** the old one, so the older updates are thrown away.

Run the cell below. It feeds the same 9 updates (3 parcels × 3 scans) into both policies and prints what each one keeps.

In [ ]:
# Run this cell. Nothing to edit.
small = list(feed(parcels=3, updates=3, proof=False))
print("The feed, oldest first:")
for u in small:
    print(f"  {u[0]}  {u[1]:<14} at minute {u[2]}")
every_state = keep(small, "every")
latest_state = keep(small, "latest")
print()
print(f'policy "every" keeps {len(every_state)} entries (the whole list above).')
print(f'policy "latest" keeps {len(latest_state)} entries:')
for parcel, u in latest_state.items():
    print(f"  {parcel} -> {u[1]} at minute {u[2]}")

### Your answer — Q1

1. A customer calls: "When was parcel P000001 picked up?" Which policy can still answer that, and why can't the other?
2. In one sentence: what does `"latest"` discard, and what does it gain in exchange?

[replace]

## 2. Experiment 1: the update policy

The baseline feed is **20,000 parcels with 4 updates each** (80,000 updates), status only, no proof. Trial **1A** keeps every update. Trial **1B** keeps the latest status. Everything else stays the same, so **the policy is the only factor**.

Then you choose a **busier feed**: more updates per parcel (a number from 5 to 12) with the same 20,000 parcels. Trials **1C** and **1D** run it under each policy, so each one changes **only** the update count compared with 1A or 1B.

### Your answer — Q2: predict before you run

1. Trials 1A and 1B: how many entries will each keep? Which will use more memory, and about how many times more?
2. When the updates per parcel go up, which policy's memory will grow, and which will stay about the same? Why?

[replace]

In [ ]:
# Experiment 1, trials 1A and 1B. Nothing to edit: only the policy differs.
trial("1A", parcels=20_000, updates=4, policy="every", proof=False)
print()
trial("1B", parcels=20_000, updates=4, policy="latest", proof=False)

In [ ]:
# Experiment 1, trials 1C and 1D: your busier feed.
# Replace None with a whole number of updates per parcel from 5 to 12, then run this cell.
MY_UPDATES = None

if not isinstance(MY_UPDATES, int) or isinstance(MY_UPDATES, bool) or not 5 <= MY_UPDATES <= 12:
    raise ValueError("Set MY_UPDATES to a whole number from 5 to 12 (no quotes), for example MY_UPDATES = 8")
trial("1C", parcels=20_000, updates=MY_UPDATES, policy="every", proof=False)
print()
trial("1D", parcels=20_000, updates=MY_UPDATES, policy="latest", proof=False)

### Your answer — Q3: observe and explain

1. Record entries kept and measured MB for 1A, 1B, 1C and 1D (the printed rows have them).
2. Were your Q2 predictions supported or overturned? Explain **why** one policy grows with the number of updates and the other does not, using what each one keeps.
3. Compare entries kept with measured bytes. Did measured memory grow in the same proportion as entries? Say what that tells you about cost per entry.

**1A:** [replace] **1B:** [replace] **1C:** [replace] **1D:** [replace]

**Predictions supported or overturned, and why:** [replace]

**Entries versus measured bytes:** [replace]

## 3. Experiment 2: the detail level

Now hold the parcel count (20,000), the updates (4) and the policy (`"latest"`) fixed, exactly as in trial 1B. Change **only** the detail: `proof=True` attaches a **1,000 B** proof-of-delivery payload to each parcel's *delivered* update.

### Your answer — Q4: predict before you run

With 20,000 parcels, how many extra MB will proof add over trial 1B? Will the extra per parcel be exactly 1,000 B, or more? Why?

[replace]

In [ ]:
# Experiment 2, trial 2A. Replace None with True, so that only the detail changes from trial 1B. Then run.
trial("2A", parcels=20_000, updates=4, policy="latest", proof=None)

### Your answer — Q5: observe and explain

1. From the 2A row and the 1B row, compute the extra measured bytes **per parcel** that proof added. Show the subtraction and the division.
2. The payload counted on paper is 1,000 B per parcel. Is the measured extra bigger? Name what else Python keeps for each payload.
3. Note the 2A **measured per parcel** figure. You will use it in Q6.

**Extra per parcel:** [replace]

**Paper versus measured:** [replace]

## 4. Experiment 3: find the boundary

Keep the representation from trial 2A fixed (**`"latest"`, proof on, 4 updates**). The only setting you change now is the **number of active parcels**. Your task is to **bracket** the memory limit: find a count that fits and a count that does not, **no more than 250 parcels apart**.

### Your answer — Q6: predict the boundary before your first trial

Use the measured-per-parcel figure from 2A: about how many parcels fit in 10,000,000 B? Show the division.

[replace]

**How to run Experiment 3:** in the cell below, set `PARCELS` to a count and `I_PREDICT` to `"fits"` or `"over"` **before** each run. Run it. Then change both and run the same cell again. Each run is saved as the next trial (3A, 3B, …), and the report cell after it shows them all. Expect three to five runs.

In [ ]:
# Experiment 3. Set both lines, run, then change both and run this same cell again.
PARCELS = None          # a whole number from 1000 to 40000
I_PREDICT = None        # "fits" or "over", decided BEFORE you run

if I_PREDICT not in ("fits", "over"):
    raise ValueError('Set I_PREDICT to "fits" or "over" (with the quotes) before you run.')
name = f"3{chr(ord('A') + sum(1 for t in TRIALS if t.startswith('3')))}"
if name > "3J":
    raise ValueError("That is ten boundary trials. Stop and write up the bracket you have.")
row = trial(name, parcels=PARCELS, updates=4, policy="latest", proof=True)
row["predicted"] = I_PREDICT
actual = "fits" if row["measured"] <= LIMIT_BYTES else "over"
print(f"  You predicted {I_PREDICT!r}; it {actual}.")

In [ ]:
# Boundary report. Run after each Experiment 3 trial (or once at the end). Nothing to edit.
runs = [(n, r) for n, r in TRIALS.items() if n.startswith("3")]
if not runs:
    print("No Experiment 3 trials yet. Run the cell above first.")
else:
    for n, r in runs:
        verdict = "fits" if r["measured"] <= LIMIT_BYTES else "over"
        print(f"  {n}: {r['parcels']:>6,} parcels  {r['measured']:>12,} B  {verdict:<5} (predicted {r['predicted']})")
    fit = [r["parcels"] for _, r in runs if r["measured"] <= LIMIT_BYTES]
    over = [r["parcels"] for _, r in runs if r["measured"] > LIMIT_BYTES]
    if fit and over and max(fit) < min(over):
        gap = min(over) - max(fit)
        print(f"Bracket: {max(fit):,} fits, {min(over):,} is over, {gap:,} parcels apart.")
        print("Bracket found." if gap <= 250 else "Narrow it: try a count between those two.")
    else:
        print("You need at least one count that fits and one that is over. Keep going.")

### Your answer — Q7: the boundary

1. Your bracket: the largest count that fits and the smallest that is over, with their measured bytes.
2. How close was your Q6 prediction? If it was off, say why a per-parcel estimate can miss.
3. How many of your fits/over predictions were right?

[replace]

## 5. Decide

The company expects **up to 7,000 active parcels** at peak, with about 4 updates each, under the 10 MB limit. Choose the setting you would recommend: a **policy** and **proof** on or off. Enter them below and run the check. You may run it again with a different choice; the last run is your recommendation.

In [ ]:
# Decision check. Replace both None values with your recommended setting, then run.
MY_POLICY = None        # "every" or "latest"
MY_PROOF = None         # True or False

trial("D", parcels=7_000, updates=4, policy=MY_POLICY, proof=MY_PROOF)
print()
show_trials()

### Your answer — Q8: your recommendation

1. Your setting, its measured MB at 7,000 parcels, and whether it fits.
2. The information tradeoff: what can the company **not** do with your choice that another setting would allow? (Q1 and Experiment 2 are the evidence.)
3. Which trial results justify the choice? Name at least two trials.
4. One sentence: one thing this measurement does **not** establish (for example a real platform, speed, other Python versions, or real photos).

[replace]

## Submit your shared Colab URL

Check that Q1–Q8 are filled in, the trial outputs are visible (1A–1D, 2A, your Experiment 3 runs with the boundary report, and D), and your notebook has no unanswered `[replace]`.

1. Save your own Drive copy in Colab. Keep the cell outputs saved with it; if **Edit → Notebook settings → Omit code cell output when saving this notebook** is selected, clear it, save again, and reopen your copy to confirm the outputs remain.
2. Click **Share** and set **General access → Anyone with the link → Viewer**. If your account restricts this, share directly with your instructor as a Viewer instead.
3. Copy **your completed copy's sharing link** and submit that URL in Canvas by the deadline shown there.
4. Verify access: for an "Anyone with the link" link, open it in a private/incognito window and check that your answers and outputs are visible. For a restricted link, confirm that your instructor has Viewer access.

Submit the URL of your own completed notebook. The starter GitHub/Colab URL and a local file path are not your submission.